# AlphaGPT 项目学习教程

## 学习路线图

```
阶段1: 数据管道       →  数据从哪来、怎么存
阶段2: 特征工程       →  原始数据变成模型能理解的数字
阶段3: 模型架构       →  AlphaGPT 如何生成交易公式
阶段4: 训练与回测     →  强化学习 + 公式评分
阶段5: 策略执行       →  信号变成真实交易
```

```
[Tushare API] → [DataManager] → [PostgreSQL]
                                      ↓
                              [FeatureEngineer]  → 6维特征张量
                                      ↓
                              [AlphaGPT Transformer] → 生成公式 Token 序列
                                      ↓
                              [StackVM 虚拟机] → 执行公式，产生信号
                                      ↓
                              [MemeBacktest] → 回测评分
                                      ↓
                              [StrategyRunner] → 实盘执行
```

**前置要求**: Python 3.10+, PyTorch, pandas, numpy, tushare

---

In [1]:
# 环境准备：确保在项目根目录下运行
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"项目根目录: {os.path.abspath('..')}")

PyTorch: 2.10.0+cu128
Device: cuda
项目根目录: /home/gj/AlphaGPT


---
# 阶段 1: 数据管道

**学习目标**: 理解数据从 Tushare API → PostgreSQL 的完整流程

**涉及文件**:
- `data_pipeline/config.py` — 配置中心
- `data_pipeline/providers/tushare.py` — A股数据获取
- `data_pipeline/db_manager.py` — 数据库操作
- `data_pipeline/data_manager.py` — 编排器

## 1.1 配置系统

所有配置集中在一个 Config 类中，从 `.env` 文件读取敏感信息（Token、数据库密码等）。

In [2]:
# 查看配置结构
from data_pipeline.config import Config, DataSourceMode

# DataSourceMode 是一个枚举，定义了两种数据源
print("可用的数据源模式:")
for mode in DataSourceMode:
    print(f"  - {mode.name}: {mode.value}")

print(f"\n当前模式: {Config.DATA_SOURCE_MODE}")
print(f"股票池类型: {Config.ASTOCK_POOL_TYPE}")
print(f"最小市值: {Config.ASTOCK_MIN_MARKET_CAP / 1e8:.0f} 亿元")
print(f"历史数据起始: {Config.ASTOCK_START_DATE}")
print(f"并发数: {Config.CONCURRENCY}")

可用的数据源模式:
  - SOLANA: solana
  - ASTOCK: astock

当前模式: DataSourceMode.ASTOCK
股票池类型: hs300
最小市值: 10 亿元
历史数据起始: 20200101
并发数: 20


### 理解要点

Config 是一个纯静态类，所有属性都是**类属性**而非实例属性：

```python
class Config:
    TUSHARE_TOKEN = os.getenv("TUSHARE_TOKEN", "")  # 从环境变量读取
    ASTOCK_POOL_TYPE = "hs300"                        # 默认沪深300
```

好处：任何地方写 `Config.TUSHARE_TOKEN` 就能拿到值，不需要传参。

## 1.2 Provider 设计模式

系统使用**抽象基类 + 工厂模式**来支持多数据源：

```
DataProvider (抽象基类)
├── BirdeyeProvider   (Solana 链上数据)
└── TushareProvider   (A股数据)
```

不管什么数据源，对外只暴露**两个方法**：

In [3]:
# 查看抽象基类的接口定义
import inspect
from data_pipeline.providers.base import DataProvider

print("DataProvider 接口定义:")
print(inspect.getsource(DataProvider))

DataProvider 接口定义:
class DataProvider(ABC):    
    @abstractmethod
    async def get_trending_tokens(self, limit: int):
        pass

    @abstractmethod
    async def get_token_history(self, session, address: str, days: int):
        pass



In [4]:
# 动手实验：连接 Tushare 获取数据
import tushare as ts

pro = ts.pro_api()
pro._DataApi__token = Config.TUSHARE_TOKEN
pro._DataApi__http_url = Config.TUSHARE_API_URL

# 实验1: 获取沪深300成分股（前10只）
print("=== 沪深300 成分股（前10只） ===")
df_weight = pro.index_weight(index_code='000300.SH', limit=10)
print(df_weight[['con_code', 'trade_date', 'weight']])

print("\n含义解读:")
print("  con_code:   成分股代码")
print("  trade_date: 成分股日期")
print("  weight:     在指数中的权重(%)")

=== 沪深300 成分股（前10只） ===
    con_code trade_date  weight
0  300750.SZ   20260130   3.604
1  600519.SH   20260130   3.415
2  300308.SZ   20260130   2.808
3  601318.SH   20260130   2.770
4  601899.SH   20260130   2.574
5  600036.SH   20260130   1.864
6  300502.SZ   20260130   1.623
7  000333.SZ   20260130   1.488
8  600900.SH   20260130   1.256
9  601166.SH   20260130   1.233

含义解读:
  con_code:   成分股代码
  trade_date: 成分股日期
  weight:     在指数中的权重(%)


In [5]:
# 实验2: 获取单只股票的日线数据
print("=== 平安银行 (000001.SZ) 最近5天日线 ===")
df_daily = pro.daily(ts_code='000001.SZ', limit=5)
print(df_daily[['trade_date', 'open', 'high', 'low', 'close', 'vol', 'amount']])

print("\n字段含义:")
print("  vol:    成交量 (单位: 手, 1手=100股)")
print("  amount: 成交额 (单位: 千元)")

# 系统中的单位转换
row = df_daily.iloc[0]
print(f"\n单位转换示例 (以 {row['trade_date']} 为例):")
print(f"  成交量: {row['vol']:.0f} 手 → {row['vol'] * 100:.0f} 股")
print(f"  成交额: {row['amount']:.0f} 千元 → {row['amount'] * 1000:.0f} 元 = {row['amount'] * 1000 / 1e8:.2f} 亿元")

=== 平安银行 (000001.SZ) 最近5天日线 ===
  trade_date   open   high    low  close         vol       amount
0   20260202  10.82  11.03  10.80  10.86  1223173.02  1337643.445
1   20260130  10.94  11.03  10.83  10.83  1091754.97  1191181.198
2   20260129  10.84  10.98  10.66  10.96  1852993.83  2001845.208
3   20260128  10.94  10.99  10.82  10.84  1438766.03  1570012.850
4   20260127  10.96  11.03  10.93  10.94   894091.21   981119.838

字段含义:
  vol:    成交量 (单位: 手, 1手=100股)
  amount: 成交额 (单位: 千元)

单位转换示例 (以 20260202 为例):
  成交量: 1223173 手 → 122317302 股
  成交额: 1337643 千元 → 1337643445 元 = 13.38 亿元


In [6]:
# 实验3: 获取市值数据
print("=== 每日基本指标 ===")
df_basic = pro.daily_basic(ts_code='000001.SZ', limit=3, fields='ts_code,trade_date,total_mv,pe,turnover_rate_f')
print(df_basic)

print("\n字段含义:")
print("  total_mv:        总市值 (单位: 万元)")
print("  pe:              市盈率")
print("  turnover_rate_f: 换手率(自由流通股本)")

if not df_basic.empty:
    mv = df_basic.iloc[0]['total_mv']
    print(f"\n市值转换: {mv:.0f} 万元 → {mv * 10000 / 1e8:.2f} 亿元")
    print(f"系统存储为 fdv = {mv * 10000:.0f} 元")

=== 每日基本指标 ===
     ts_code trade_date      total_mv      pe  turnover_rate_f
0  000001.SZ   20260202  2.107483e+07  4.7351           1.4989
1  000001.SZ   20260130  2.101661e+07  4.7220           1.3379
2  000001.SZ   20260129  2.126889e+07  4.7787           2.2707

字段含义:
  total_mv:        总市值 (单位: 万元)
  pe:              市盈率
  turnover_rate_f: 换手率(自由流通股本)

市值转换: 21074827 万元 → 2107.48 亿元
系统存储为 fdv = 210748271630 元


## 1.3 字段映射（关键）

| A股原始字段 | Tushare 单位 | 系统字段 | 系统单位 | 转换公式 |
|------------|-------------|---------|---------|----------|
| ts_code | 600519.SH | `address` | 字符串 | 直接映射 |
| vol | 手 (100股) | `volume` | 股 | `vol * 100` |
| amount | 千元 | `liquidity` | 元 | `amount * 1000` |
| total_mv | 万元 | `fdv` | 元 | `total_mv * 10000` |
| open/high/low/close | 元 | 同名 | 元 | 直接映射 |

## 1.4 数据管道编排

DataManager 是整个数据管道的**编排器**，它的 `pipeline_sync_daily()` 方法执行以下流程：

```
Step 1: get_trending_tokens()  → 获取候选股票
Step 2: 过滤 (市值/流动性)     → 筛选合格股票
Step 3: upsert_tokens()        → 写入 tokens 表
Step 4: get_token_history()    → 批量获取历史K线 → 写入 ohlcv 表
```

In [10]:
# 动手实验：模拟数据管道的每一步
import asyncio
from data_pipeline.providers.tushare import TushareProvider

provider = TushareProvider()

# Step 1: 获取候选股票
print("Step 1: 获取沪深300成分股...")
tokens = await provider.get_trending_tokens(limit=10)
print(f"获取到 {len(tokens)} 只股票:\n")
for t in tokens[:5]:
    print(f"  {t['address']:>10} | {t['name']:<6} | 市值: {t['fdv']/1e8:>8.2f} 亿")

# Step 2: 过滤
min_cap = Config.ASTOCK_MIN_MARKET_CAP
filtered = [t for t in tokens if t['fdv'] >= min_cap]
print(f"\nStep 2: 过滤后剩余 {len(filtered)}/{len(tokens)} 只 (市值 >= {min_cap/1e8:.0f}亿)")

2026-02-02 22:48:11.105 | INFO     | data_pipeline.providers.tushare:__init__:40 - TushareProvider initialized with API: http://1w2b.xiximiao.com/dataapi


Step 1: 获取沪深300成分股...


2026-02-02 22:48:14.474 | INFO     | data_pipeline.providers.tushare:get_trending_tokens:57 - Tushare fetched 296 stocks from pool 'hs300'


获取到 10 只股票:

   000001.SZ | 平安银行   | 市值:  2107.48 亿
   000002.SZ | 万科Ａ    | 市值:   558.36 亿
   000063.SZ | 中兴通讯   | 市值:  1773.73 亿
   000100.SZ | TCL科技  | 市值:   988.04 亿
   000157.SZ | 中联重科   | 市值:   747.23 亿

Step 2: 过滤后剩余 10/10 只 (市值 >= 10亿)


In [11]:
# Step 4: 获取单只股票的历史数据
if filtered:
    sample = filtered[0]
    print(f"Step 4: 获取 {sample['name']} ({sample['address']}) 的历史数据...")
    
    history = await provider.get_token_history(None, sample['address'])
    print(f"获取到 {len(history)} 条记录\n")
    
    # 查看数据结构
    print("数据结构 (每条记录的字段):")
    fields = ['time', 'address', 'open', 'high', 'low', 'close', 'volume', 'liquidity', 'fdv', 'source']
    if history:
        for i, (field, val) in enumerate(zip(fields, history[0])):
            print(f"  [{i}] {field:>10} = {val}")
        
        print(f"\n最早: {history[-1][0].strftime('%Y-%m-%d')}")
        print(f"最新: {history[0][0].strftime('%Y-%m-%d')}")

Step 4: 获取 平安银行 (000001.SZ) 的历史数据...
获取到 1476 条记录

数据结构 (每条记录的字段):
  [0]       time = 2026-02-02 00:00:00
  [1]    address = 000001.SZ
  [2]       open = 10.82
  [3]       high = 11.03
  [4]        low = 10.8
  [5]      close = 10.86
  [6]     volume = 122317302.0
  [7]  liquidity = 1337643445.0
  [8]        fdv = 0.0
  [9]     source = tushare

最早: 2020-01-02
最新: 2026-02-02


## 1.5 数据库存储结构

```sql
-- tokens 表: 股票基本信息
CREATE TABLE tokens (
    address TEXT PRIMARY KEY,    -- '000001.SZ'
    symbol  TEXT,                -- '000001'
    name    TEXT,                -- '平安银行'
    decimals INT,               -- 2
    chain   TEXT,               -- 'astock'
    last_updated TIMESTAMP
);

-- ohlcv 表: 时序行情 (TimescaleDB 超表)
CREATE TABLE ohlcv (
    time      TIMESTAMP NOT NULL,   -- 交易日期
    address   TEXT NOT NULL,         -- 股票代码
    open      DOUBLE PRECISION,      -- 开盘价
    high      DOUBLE PRECISION,      -- 最高价
    low       DOUBLE PRECISION,      -- 最低价
    close     DOUBLE PRECISION,      -- 收盘价
    volume    DOUBLE PRECISION,      -- 成交量(股)
    liquidity DOUBLE PRECISION,      -- 成交额(元)
    fdv       DOUBLE PRECISION,      -- 总市值(元)
    source    TEXT,                  -- 'tushare'
    PRIMARY KEY (time, address)
);
```

**为什么用 TimescaleDB?** 普通 PostgreSQL 查询百万行需要几秒，TimescaleDB 自动按时间分区，同样查询只需 0.1 秒。

---

# 阶段 2: 特征工程

**学习目标**: 理解原始 OHLCV 如何变成模型输入的 6 维特征张量

**涉及文件**:
- `model_core/factors.py` — 特征计算
- `model_core/data_loader.py` — 数据加载

## 2.1 数据加载：从数据库到张量

```
SQL 表 (长表)                Pivot 转换 (宽表)              PyTorch 张量
┌──────────┬─────────┐      ┌──────────┬─────────┐       shape: [N, T]
│time      │address  │close │          │000001.SZ│000002 │       ↓   ↓
│2025-01-01│000001.SZ│11.6  │2025-01-01│  11.6   │ 6.8  │  N只股票  T天
│2025-01-01│000002.SZ│ 6.8  │2025-01-02│  11.5   │ 6.9  │
│2025-01-02│000001.SZ│11.5  │          │         │      │
│2025-01-02│000002.SZ│ 6.9  │          │         │      │
└──────────┴─────────┘      └──────────┴─────────┘
```

In [12]:
# 用模拟数据演示整个特征工程流程
# (不依赖数据库，方便任何人运行)

torch.manual_seed(42)
np.random.seed(42)

N_STOCKS = 5     # 5 只股票
T_DAYS = 100     # 100 个交易日

# 模拟 OHLCV 数据
# 使用随机游走模拟股价
returns = np.random.normal(0.001, 0.02, (N_STOCKS, T_DAYS))  # 日均涨0.1%, 波动2%
close_np = 10.0 * np.exp(np.cumsum(returns, axis=1))  # 从10元起

# 由 close 生成 open/high/low
noise = np.random.uniform(0.005, 0.02, (N_STOCKS, T_DAYS))
high_np = close_np * (1 + noise)
low_np = close_np * (1 - noise)
open_np = close_np * (1 + np.random.normal(0, 0.005, (N_STOCKS, T_DAYS)))

# 成交量和成交额
volume_np = np.random.uniform(50_000_000, 200_000_000, (N_STOCKS, T_DAYS))  # 5千万~2亿股
liquidity_np = volume_np * close_np * 0.5  # 近似成交额
fdv_np = np.random.uniform(50e9, 500e9, (N_STOCKS, 1)) * np.ones((1, T_DAYS))  # 市值500~5000亿

# 转为 PyTorch 张量
raw_data = {
    'open':      torch.tensor(open_np, dtype=torch.float32),
    'high':      torch.tensor(high_np, dtype=torch.float32),
    'low':       torch.tensor(low_np, dtype=torch.float32),
    'close':     torch.tensor(close_np, dtype=torch.float32),
    'volume':    torch.tensor(volume_np, dtype=torch.float32),
    'liquidity': torch.tensor(liquidity_np, dtype=torch.float32),
    'fdv':       torch.tensor(fdv_np, dtype=torch.float32),
}

print("模拟数据已生成:")
for k, v in raw_data.items():
    print(f"  {k:>10}: shape={list(v.shape)}, 范围=[{v.min():.2f}, {v.max():.2f}]")

模拟数据已生成:
        open: shape=[5, 100], 范围=[8.29, 13.85]
        high: shape=[5, 100], 范围=[8.41, 13.90]
         low: shape=[5, 100], 范围=[8.16, 13.62]
       close: shape=[5, 100], 范围=[8.29, 13.76]
      volume: shape=[5, 100], 范围=[50482740.00, 199752128.00]
   liquidity: shape=[5, 100], 范围=[232746752.00, 1240176256.00]
         fdv: shape=[5, 100], 范围=[98611593216.00, 411163557888.00]


## 2.2 逐个理解 6 维特征

| # | 名称 | 公式 | 直觉含义 |
|---|------|------|----------|
| 0 | RET | `log(close[t] / close[t-1])` | 今天涨了还是跌了 |
| 1 | LIQ_SCORE | `clamp(liquidity/fdv * 4, 0, 1)` | 换手活跃度 |
| 2 | PRESSURE | `tanh((close-open)/(high-low) * 3)` | 阳线还是阴线 |
| 3 | FOMO | 成交量二阶差分 | 资金在加速涌入还是撤退 |
| 4 | DEV | `(close - MA20) / MA20` | 偏离均线多远 |
| 5 | LOG_VOL | `log(1 + volume)` | 成交量大小 |

In [13]:
# 特征 0: RET（对数收益率）

c = raw_data['close']
c_prev = torch.roll(c, 1, dims=1)  # 昨天的收盘价
ret = torch.log(c / (c_prev + 1e-9))  # 对数收益率

# 用第一只股票的前10天来演示
stock_0_close = c[0, :10].numpy()
stock_0_ret = ret[0, 1:10].numpy()  # 第0天没有前一天，跳过

print("=== 特征 0: RET (对数收益率) ===")
print("\n公式: ret[t] = log(close[t] / close[t-1])")
print("\n逐日计算:")
for i in range(1, min(8, len(stock_0_close))):
    pct = (stock_0_close[i] / stock_0_close[i-1] - 1) * 100
    print(f"  Day {i}: close={stock_0_close[i]:.4f} / prev={stock_0_close[i-1]:.4f}"
          f" → ret={stock_0_ret[i-1]:+.4f} ({pct:+.2f}%)")

print("\n为什么用 log 而不是简单百分比?")
print("  因为 log 收益率具有可加性: log(P2/P0) = log(P2/P1) + log(P1/P0)")
print("  这对时序模型计算非常方便")

=== 特征 0: RET (对数收益率) ===

公式: ret[t] = log(close[t] / close[t-1])

逐日计算:
  Day 1: close=10.0921 / prev=10.1099 → ret=-0.0018 (-0.18%)
  Day 2: close=10.2339 / prev=10.0921 → ret=+0.0140 (+1.41%)
  Day 3: close=10.5610 / prev=10.2339 → ret=+0.0315 (+3.20%)
  Day 4: close=10.5222 / prev=10.5610 → ret=-0.0037 (-0.37%)
  Day 5: close=10.4835 / prev=10.5222 → ret=-0.0037 (-0.37%)
  Day 6: close=10.8307 / prev=10.4835 → ret=+0.0326 (+3.31%)
  Day 7: close=11.0092 / prev=10.8307 → ret=+0.0163 (+1.65%)

为什么用 log 而不是简单百分比?
  因为 log 收益率具有可加性: log(P2/P0) = log(P2/P1) + log(P1/P0)
  这对时序模型计算非常方便


In [14]:
# 特征 1: LIQ_SCORE（流动性评分）

liq = raw_data['liquidity']
fdv = raw_data['fdv']

liq_score = torch.clamp(liq / (fdv + 1e-6) * 4.0, 0.0, 1.0)

print("=== 特征 1: LIQ_SCORE (流动性评分) ===")
print("\n公式: liq_score = clamp(日成交额 / 总市值 * 4, 0, 1)")
print("\n逐股计算 (第1天):")
for i in range(N_STOCKS):
    l = liq[i, 0].item()
    f = fdv[i, 0].item()
    s = liq_score[i, 0].item()
    print(f"  Stock {i}: 成交额={l/1e8:.1f}亿 / 市值={f/1e8:.0f}亿 * 4 = {s:.4f}")

print("\n解读: 越接近1说明换手越活跃 (资金关注度高)")

=== 特征 1: LIQ_SCORE (流动性评分) ===

公式: liq_score = clamp(日成交额 / 总市值 * 4, 0, 1)

逐股计算 (第1天):
  Stock 0: 成交额=3.1亿 / 市值=986亿 * 4 = 0.0124
  Stock 1: 成交额=5.0亿 / 市值=2025亿 * 4 = 0.0099
  Stock 2: 成交额=3.0亿 / 市值=4112亿 * 4 = 0.0029
  Stock 3: 成交额=8.4亿 / 市值=3074亿 * 4 = 0.0109
  Stock 4: 成交额=6.4亿 / 市值=2807亿 * 4 = 0.0091

解读: 越接近1说明换手越活跃 (资金关注度高)


In [15]:
# 特征 2: PRESSURE（K线压力）

o = raw_data['open']
h = raw_data['high']
l = raw_data['low']

range_hl = h - l + 1e-9
body = c - o
pressure = torch.tanh(body / range_hl * 3.0)

print("=== 特征 2: PRESSURE (K线压力) ===")
print("\n公式: pressure = tanh((close - open) / (high - low) * 3)")
print("\n逐日计算 (第一只股票):")
for i in range(min(6, T_DAYS)):
    oi, hi, li, ci = o[0,i].item(), h[0,i].item(), l[0,i].item(), c[0,i].item()
    p = pressure[0, i].item()
    bar_type = "阳线" if ci > oi else "阴线" if ci < oi else "十字"
    print(f"  Day {i}: O={oi:.2f} H={hi:.2f} L={li:.2f} C={ci:.2f}"
          f" → pressure={p:+.3f} ({bar_type})")

print("\n解读:")
print("  +1 附近 → 强势阳线 (收盘价接近最高价)")
print("  -1 附近 → 强势阴线 (收盘价接近最低价)")
print("   0 附近 → 十字星 (多空平衡)")

=== 特征 2: PRESSURE (K线压力) ===

公式: pressure = tanh((close - open) / (high - low) * 3)

逐日计算 (第一只股票):
  Day 0: O=10.10 H=10.26 L=9.96 C=10.11 → pressure=+0.085 (阳线)
  Day 1: O=10.10 H=10.23 L=9.96 C=10.09 → pressure=-0.081 (阴线)
  Day 2: O=10.30 H=10.34 L=10.13 C=10.23 → pressure=-0.704 (阴线)
  Day 3: O=10.52 H=10.77 L=10.35 C=10.56 → pressure=+0.300 (阳线)
  Day 4: O=10.54 H=10.67 L=10.37 C=10.52 → pressure=-0.194 (阴线)
  Day 5: O=10.46 H=10.57 L=10.39 C=10.48 → pressure=+0.332 (阳线)

解读:
  +1 附近 → 强势阳线 (收盘价接近最高价)
  -1 附近 → 强势阴线 (收盘价接近最低价)
   0 附近 → 十字星 (多空平衡)


In [16]:
# 特征 3: FOMO（成交量加速度）

v = raw_data['volume']
v_prev = torch.roll(v, 1, dims=1)
vol_chg = (v - v_prev) / (v_prev + 1.0)         # 一阶变化率
vol_chg_prev = torch.roll(vol_chg, 1, dims=1)    # 前一期的变化率
fomo = torch.clamp(vol_chg - vol_chg_prev, -5.0, 5.0)  # 二阶差分 = 加速度

print("=== 特征 3: FOMO (成交量加速度) ===")
print("\n公式: fomo = (vol_chg[t] - vol_chg[t-1])")
print("  其中 vol_chg = (vol[t] - vol[t-1]) / vol[t-1]")
print("\n逐日计算 (第一只股票):")
for i in range(2, min(8, T_DAYS)):
    vi = v[0, i].item()
    vi_1 = v[0, i-1].item()
    vi_2 = v[0, i-2].item()
    chg_now = (vi - vi_1) / vi_1
    chg_prev = (vi_1 - vi_2) / vi_2
    acc = chg_now - chg_prev
    label = "加速涌入" if acc > 0.1 else "加速撤退" if acc < -0.1 else "平稳"
    print(f"  Day {i}: vol={vi/1e6:.1f}M, 变化率={chg_now:+.3f}, 加速度={acc:+.3f} ({label})")

print("\n解读:")
print("  正值 → 资金在加速涌入 (FOMO情绪)")
print("  负值 → 资金在加速撤离 (恐慌情绪)")

=== 特征 3: FOMO (成交量加速度) ===

公式: fomo = (vol_chg[t] - vol_chg[t-1])
  其中 vol_chg = (vol[t] - vol[t-1]) / vol[t-1]

逐日计算 (第一只股票):
  Day 2: vol=193.5M, 变化率=+2.734, 加速度=+2.880 (加速涌入)
  Day 3: vol=160.6M, 变化率=-0.170, 加速度=-2.904 (加速撤退)
  Day 4: vol=103.0M, 变化率=-0.359, 加速度=-0.189 (加速撤退)
  Day 5: vol=94.5M, 变化率=-0.083, 加速度=+0.276 (加速涌入)
  Day 6: vol=102.5M, 变化率=+0.084, 加速度=+0.167 (加速涌入)
  Day 7: vol=166.2M, 变化率=+0.622, 加速度=+0.538 (加速涌入)

解读:
  正值 → 资金在加速涌入 (FOMO情绪)
  负值 → 资金在加速撤离 (恐慌情绪)


In [17]:
# 特征 4: DEV（价格偏离度）

window = 20
# 手动计算 MA20
pad = torch.zeros((N_STOCKS, window - 1))
c_padded = torch.cat([pad, c], dim=1)
ma20 = c_padded.unfold(1, window, 1).mean(dim=-1)
dev = (c - ma20) / (ma20 + 1e-9)

print("=== 特征 4: DEV (价格偏离度) ===")
print("\n公式: dev = (close - MA20) / MA20")
print("\n逐日计算 (第一只股票, 从第20天起):")
for i in range(20, min(28, T_DAYS)):
    ci = c[0, i].item()
    mi = ma20[0, i].item()
    di = dev[0, i].item()
    label = "超涨" if di > 0.05 else "超跌" if di < -0.05 else "正常"
    print(f"  Day {i}: close={ci:.3f}, MA20={mi:.3f}, dev={di:+.4f} ({di*100:+.2f}%, {label})")

print("\n解读:")
print("  > 0  → 价格在均线上方 (偏多)")
print("  < 0  → 价格在均线下方 (偏空)")
print("  极值 → 可能超涨/超跌，有回调/反弹风险")

=== 特征 4: DEV (价格偏离度) ===

公式: dev = (close - MA20) / MA20

逐日计算 (第一只股票, 从第20天起):
  Day 20: close=9.820, MA20=10.411, dev=-0.0568 (-5.68%, 超跌)
  Day 21: close=9.785, MA20=10.396, dev=-0.0587 (-5.87%, 超跌)
  Day 22: close=9.808, MA20=10.374, dev=-0.0546 (-5.46%, 超跌)
  Day 23: close=9.542, MA20=10.323, dev=-0.0757 (-7.57%, 超跌)
  Day 24: close=9.448, MA20=10.270, dev=-0.0800 (-8.00%, 超跌)
  Day 25: close=9.479, MA20=10.220, dev=-0.0725 (-7.25%, 超跌)
  Day 26: close=9.272, MA20=10.142, dev=-0.0857 (-8.57%, 超跌)
  Day 27: close=9.352, MA20=10.059, dev=-0.0703 (-7.03%, 超跌)

解读:
  > 0  → 价格在均线上方 (偏多)
  < 0  → 价格在均线下方 (偏空)
  极值 → 可能超涨/超跌，有回调/反弹风险


In [18]:
# 特征 5: LOG_VOL（对数成交量）

log_vol = torch.log1p(v)

print("=== 特征 5: LOG_VOL (对数成交量) ===")
print("\n公式: log_vol = log(1 + volume)")
print("\n为什么取对数?")
print("  小盘股日成交: 1,000,000 股  → log1p = 13.8")
print("  大盘股日成交: 100,000,000 股 → log1p = 18.4")
print("  差距从 100倍 缩小为 1.3倍，模型更容易学习")

print(f"\n实际数据范围: [{log_vol.min():.2f}, {log_vol.max():.2f}]")

=== 特征 5: LOG_VOL (对数成交量) ===

公式: log_vol = log(1 + volume)

为什么取对数?
  小盘股日成交: 1,000,000 股  → log1p = 13.8
  大盘股日成交: 100,000,000 股 → log1p = 18.4
  差距从 100倍 缩小为 1.3倍，模型更容易学习

实际数据范围: [17.74, 19.11]


## 2.3 鲁棒归一化

特征计算后，需要归一化让不同特征在同一尺度上。

AlphaGPT 使用**鲁棒归一化 (Robust Normalization)**，比常见的 Z-score 更抗异常值。

In [19]:
# 对比 Z-score 和 Robust Normalization

# 模拟带异常值的数据
data_with_outlier = torch.tensor([[1.0, 1.1, 1.2, 1.0, 1.1, 50.0, 0.9, 1.0, 1.1, 1.2]])
# 注意: 50.0 是一个异常值

# Z-score 归一化
mean = data_with_outlier.mean(dim=1, keepdim=True)
std = data_with_outlier.std(dim=1, keepdim=True)
z_norm = (data_with_outlier - mean) / (std + 1e-6)

# Robust 归一化 (AlphaGPT 使用的方法)
median = torch.nanmedian(data_with_outlier, dim=1, keepdim=True)[0]
mad = torch.nanmedian(torch.abs(data_with_outlier - median), dim=1, keepdim=True)[0] + 1e-6
robust_norm = torch.clamp((data_with_outlier - median) / mad, -5.0, 5.0)

print("原始数据:  ", [f"{x:.1f}" for x in data_with_outlier[0].tolist()])
print(f"            注意: 50.0 是异常值\n")

print(f"Z-score:    mean={mean.item():.2f}, std={std.item():.2f}")
print("  归一化后:", [f"{x:+.2f}" for x in z_norm[0].tolist()])
print(f"  问题: mean 被 50.0 严重拉偏，正常值全变成负数\n")

print(f"Robust:     median={median.item():.2f}, mad={mad.item():.4f}")
print("  归一化后:", [f"{x:+.2f}" for x in robust_norm[0].tolist()])
print(f"  优点: 正常值分布在 [-1, +1]，异常值被截断到 +5")

原始数据:   ['1.0', '1.1', '1.2', '1.0', '1.1', '50.0', '0.9', '1.0', '1.1', '1.2']
            注意: 50.0 是异常值

Z-score:    mean=5.96, std=15.47
  归一化后: ['-0.32', '-0.31', '-0.31', '-0.32', '-0.31', '+2.85', '-0.33', '-0.32', '-0.31', '-0.31']
  问题: mean 被 50.0 严重拉偏，正常值全变成负数

Robust:     median=1.10, mad=0.1000
  归一化后: ['-1.00', '+0.00', '+1.00', '-1.00', '+0.00', '+5.00', '-2.00', '-1.00', '+0.00', '+1.00']
  优点: 正常值分布在 [-1, +1]，异常值被截断到 +5


In [20]:
# 使用真实的 FeatureEngineer 计算完整特征
from model_core.factors import FeatureEngineer

feat_tensor = FeatureEngineer.compute_features(raw_data)

print("=== 特征工程最终输出 ===")
print(f"形状: {list(feat_tensor.shape)}")
print(f"  dim 0: {feat_tensor.shape[0]} 只股票")
print(f"  dim 1: {feat_tensor.shape[1]} 个特征")
print(f"  dim 2: {feat_tensor.shape[2]} 个交易日")

feature_names = ['RET', 'LIQ_SCORE', 'PRESSURE', 'FOMO', 'DEV', 'LOG_VOL']
print("\n每个特征的统计:")
for i, name in enumerate(feature_names):
    f = feat_tensor[:, i, :]
    print(f"  [{i}] {name:>10}: mean={f.mean():.4f}, std={f.std():.4f}, "
          f"range=[{f.min():.3f}, {f.max():.3f}]")

=== 特征工程最终输出 ===
形状: [5, 6, 100]
  dim 0: 5 只股票
  dim 1: 6 个特征
  dim 2: 100 个交易日

每个特征的统计:
  [0]        RET: mean=0.0196, std=1.5716, range=[-5.000, 5.000]
  [1]  LIQ_SCORE: mean=0.0126, std=0.0077, range=[0.003, 0.040]
  [2]   PRESSURE: mean=-0.0349, std=0.5026, range=[-0.997, 0.987]
  [3]       FOMO: mean=0.1505, std=1.7281, range=[-5.000, 5.000]
  [4]        DEV: mean=0.6663, std=2.1900, range=[-2.700, 5.000]
  [5]    LOG_VOL: mean=-0.2956, std=1.3380, range=[-3.586, 1.810]


In [19]:
feat_tensor

tensor([[[ 5.0000e+00,  0.0000e+00,  1.5721e+00,  ...,  7.9872e-01,
           2.8679e-01, -1.9267e-01],
         [ 1.2444e-02,  1.0606e-02,  4.0158e-02,  ...,  2.9206e-02,
           2.6028e-02,  1.4242e-02],
         [ 8.5122e-02, -8.0961e-02, -7.0390e-01,  ...,  2.9357e-01,
           4.8177e-01,  1.6074e-01],
         [ 7.6797e-01,  4.4004e-01,  5.0000e+00,  ..., -7.2747e-01,
          -4.1332e-01, -4.8125e-01],
         [ 5.0000e+00,  5.0000e+00,  5.0000e+00,  ..., -8.5626e-02,
          -2.3796e-02, -1.3485e-01],
         [-2.9683e+00, -3.5564e+00,  1.3459e+00,  ...,  6.3809e-01,
           2.0533e-01, -2.0247e+00]],

        [[-5.0000e+00, -7.1581e-01, -6.0179e-01,  ...,  1.2461e-01,
          -1.5150e-02, -1.7727e+00],
         [ 9.9290e-03,  1.5986e-02,  1.2627e-02,  ...,  1.7292e-02,
           2.2573e-02,  6.8876e-03],
         [-6.2455e-01, -2.7808e-01,  5.2905e-01,  ..., -4.9527e-02,
          -7.5752e-01,  5.0065e-02],
         [ 2.4723e+00, -1.0649e-01, -1.3830e+00,  ...

## 2.4 构造标签（目标收益率）

模型需要预测的目标是**下期开盘到下下期开盘的收益率**：

```
target_ret[t] = log(open[t+2] / open[t+1])
```

为什么不用 close-to-close？因为日线交易只能在**开盘时成交**，用 open-to-open 更准确反映实际可交易收益。

In [21]:
# 构造标签
op = raw_data['open']
t1 = torch.roll(op, -1, dims=1)  # t+1 开盘价
t2 = torch.roll(op, -2, dims=1)  # t+2 开盘价
target_ret = torch.log(t2 / (t1 + 1e-9))
target_ret[:, -2:] = 0.0  # 最后2天无法计算

print("=== 标签构造 ===")
print(f"形状: {list(target_ret.shape)} — 和特征的 [N, T] 对齐")
print("\n示例 (第一只股票):")
for t in range(3, min(8, T_DAYS)):
    o_t1 = op[0, t+1].item() if t+1 < T_DAYS else 0
    o_t2 = op[0, t+2].item() if t+2 < T_DAYS else 0
    tr = target_ret[0, t].item()
    print(f"  t={t}: open[t+1]={o_t1:.3f}, open[t+2]={o_t2:.3f}"
          f" → target_ret = log({o_t2:.3f}/{o_t1:.3f}) = {tr:+.4f} ({tr*100:+.2f}%)")

print("\n解读: target_ret > 0 表示下期开盘到下下期开盘是涨的")
print("      这就是模型要学习预测的目标")

=== 标签构造 ===
形状: [5, 100] — 和特征的 [N, T] 对齐

示例 (第一只股票):
  t=3: open[t+1]=10.542, open[t+2]=10.463 → target_ret = log(10.463/10.542) = -0.0075 (-0.75%)
  t=4: open[t+1]=10.463, open[t+2]=10.832 → target_ret = log(10.832/10.463) = +0.0347 (+3.47%)
  t=5: open[t+1]=10.832, open[t+2]=11.080 → target_ret = log(11.080/10.832) = +0.0226 (+2.26%)
  t=6: open[t+1]=11.080, open[t+2]=10.928 → target_ret = log(10.928/11.080) = -0.0138 (-1.38%)
  t=7: open[t+1]=10.928, open[t+2]=11.050 → target_ret = log(11.050/10.928) = +0.0111 (+1.11%)

解读: target_ret > 0 表示下期开盘到下下期开盘是涨的
      这就是模型要学习预测的目标


---
# 阶段 3: 模型架构

**学习目标**: 理解 AlphaGPT 如何自动生成交易公式

**涉及文件**:
- `model_core/alphagpt.py` — 模型定义
- `model_core/ops.py` — 操作符
- `model_core/vm.py` — 堆栈虚拟机

## 3.1 核心思想

**AlphaGPT 不直接预测股价，而是生成一个"交易公式"**

```
传统量化:  人写公式 → 回测 → 手动调参
AlphaGPT:  AI生成公式 → 自动回测 → 强化学习优化
```

类比：
```
GPT 生成文字:    [我] → [喜] → [欢] → [编] → [程]
AlphaGPT 生成公式: [RET] → [VOL] → [SUB] → [ABS]
                     ↓
              含义: abs(收益率 - 流动性评分)
```

## 3.2 词汇表（17个Token）

In [22]:
# 查看完整的词汇表
from model_core.ops import OPS_CONFIG

# 5 个特征 Token
features_list = ['RET', 'VOL', 'V_CHG', 'PV', 'TREND']
print("=== 词汇表 ===")
print("\n特征 Token (操作数):")
feature_desc = {
    'RET': '对数收益率',
    'VOL': '流动性评分',
    'V_CHG': '买卖压力',
    'PV': 'FOMO加速度',
    'TREND': '价格偏离度'
}
for i, name in enumerate(features_list):
    print(f"  Token {i:>2}: {name:<6} — {feature_desc[name]}")

# 12 个操作符 Token
print("\n操作符 Token (运算符):")
op_desc = {
    'ADD': 'x + y (相加)',
    'SUB': 'x - y (相减)',
    'MUL': 'x * y (相乘)',
    'DIV': 'x / y (相除)',
    'NEG': '-x (取反)',
    'ABS': '|x| (绝对值)',
    'SIGN': 'sign(x) 返回+1/-1 (方向)',
    'GATE': 'if z>0 then x else y (条件选择)',
    'JUMP': 'relu(z_score - 3) (3σ极端值检测)',
    'DECAY': 'x + 0.8*x(-1) + 0.6*x(-2) (时间衰减)',
    'DELAY1': 'x(-1) 昨天的值',
    'MAX3': 'max(x, x(-1), x(-2)) 3天最大值'
}
for i, (name, func, arity) in enumerate(OPS_CONFIG):
    token_id = i + len(features_list)
    desc = op_desc.get(name, '')
    print(f"  Token {token_id:>2}: {name:<6} (元数={arity}) — {desc}")

total = len(features_list) + len(OPS_CONFIG)
print(f"\n总词汇量: {total} 个 Token")

=== 词汇表 ===

特征 Token (操作数):
  Token  0: RET    — 对数收益率
  Token  1: VOL    — 流动性评分
  Token  2: V_CHG  — 买卖压力
  Token  3: PV     — FOMO加速度
  Token  4: TREND  — 价格偏离度

操作符 Token (运算符):
  Token  5: ADD    (元数=2) — x + y (相加)
  Token  6: SUB    (元数=2) — x - y (相减)
  Token  7: MUL    (元数=2) — x * y (相乘)
  Token  8: DIV    (元数=2) — x / y (相除)
  Token  9: NEG    (元数=1) — -x (取反)
  Token 10: ABS    (元数=1) — |x| (绝对值)
  Token 11: SIGN   (元数=1) — sign(x) 返回+1/-1 (方向)
  Token 12: GATE   (元数=3) — if z>0 then x else y (条件选择)
  Token 13: JUMP   (元数=1) — relu(z_score - 3) (3σ极端值检测)
  Token 14: DECAY  (元数=1) — x + 0.8*x(-1) + 0.6*x(-2) (时间衰减)
  Token 15: DELAY1 (元数=1) — x(-1) 昨天的值
  Token 16: MAX3   (元数=1) — max(x, x(-1), x(-2)) 3天最大值

总词汇量: 17 个 Token


## 3.3 堆栈虚拟机 (StackVM)

公式使用**逆波兰表达式 (RPN)** 执行，类似 HP 计算器：

```
数学表达式:  abs(RET - VOL)
RPN 表示:    [RET, VOL, SUB, ABS]
Token 序列:  [0, 1, 7, 11]
```

In [23]:
# 动手实验：手动执行一个公式

print("=== 手动执行公式: abs(RET - VOL) ===")
print("Token 序列: [0, 1, 7, 11]")
print("  Token 0 = RET, Token 1 = VOL, Token 7 = SUB, Token 11 = ABS\n")

# 模拟堆栈执行过程
stack = []

# Step 1: 遇到 Token 0 (RET) → 压入特征数据
ret_data = feat_tensor[:, 0, :]  # [N_stocks, T_days]
stack.append(('RET', ret_data))
print(f"Step 1: 遇到 RET  → 压栈")
print(f"  栈: [RET]   (shape: {list(ret_data.shape)})")

# Step 2: 遇到 Token 1 (VOL) → 压入特征数据
vol_data = feat_tensor[:, 1, :]  # [N_stocks, T_days]
stack.append(('VOL', vol_data))
print(f"\nStep 2: 遇到 VOL  → 压栈")
print(f"  栈: [RET, VOL]")

# Step 3: 遇到 Token 7 (SUB, 元数=2) → 弹出2个，计算，压回
_, b = stack.pop()
_, a = stack.pop()
result_sub = a - b
stack.append(('RET-VOL', result_sub))
print(f"\nStep 3: 遇到 SUB  → 弹出2个, 计算 RET - VOL, 压回")
print(f"  栈: [RET-VOL]")

# Step 4: 遇到 Token 11 (ABS, 元数=1) → 弹出1个，计算，压回
_, x = stack.pop()
result_abs = torch.abs(x)
stack.append(('|RET-VOL|', result_abs))
print(f"\nStep 4: 遇到 ABS  → 弹出1个, 计算 abs(), 压回")
print(f"  栈: [|RET-VOL|]")

print(f"\n最终结果形状: {list(result_abs.shape)} — [N_stocks, T_days] 信号矩阵")
print(f"第一只股票的前5天信号: {result_abs[0, :5].tolist()[:5]}")

=== 手动执行公式: abs(RET - VOL) ===
Token 序列: [0, 1, 7, 11]
  Token 0 = RET, Token 1 = VOL, Token 7 = SUB, Token 11 = ABS

Step 1: 遇到 RET  → 压栈
  栈: [RET]   (shape: [5, 100])

Step 2: 遇到 VOL  → 压栈
  栈: [RET, VOL]

Step 3: 遇到 SUB  → 弹出2个, 计算 RET - VOL, 压回
  栈: [RET-VOL]

Step 4: 遇到 ABS  → 弹出1个, 计算 abs(), 压回
  栈: [|RET-VOL|]

最终结果形状: [5, 100] — [N_stocks, T_days] 信号矩阵
第一只股票的前5天信号: [4.987555980682373, 0.010605965740978718, 1.5319057703018188, 3.2885258197784424, 0.21378520131111145]


In [23]:
# 使用真实的 StackVM 执行公式
from model_core.vm import StackVM

vm = StackVM()

# 公式1: abs(RET - VOL)
formula1 = [0, 1, 7, 11]  # RET, VOL, SUB, ABS
result1 = vm.execute(formula1, feat_tensor)
print("公式1: abs(RET - VOL)")
print(f"  Token: {formula1}")
print(f"  结果形状: {list(result1.shape) if result1 is not None else 'None'}")

# 公式2: sign(RET * FOMO)
formula2 = [0, 3, 8, 12]  # RET, PV(FOMO), MUL, SIGN
result2 = vm.execute(formula2, feat_tensor)
print(f"\n公式2: sign(RET * FOMO)")
print(f"  Token: {formula2}")
print(f"  含义: 收益率和FOMO同向时=+1 (做多), 反向时=-1")

# 公式3: GATE(PRESSURE, RET, DEV)
formula3 = [2, 0, 4, 13]  # V_CHG(PRESSURE), RET, TREND(DEV), GATE
result3 = vm.execute(formula3, feat_tensor)
print(f"\n公式3: if PRESSURE>0 then RET else DEV")
print(f"  Token: {formula3}")
print(f"  含义: 阳线时看收益率动量, 阴线时看偏离度")

# 无效公式: 操作数不足
formula_bad = [7, 0]  # SUB需要2个操作数，但栈里只有0个
result_bad = vm.execute(formula_bad, feat_tensor)
print(f"\n无效公式: [SUB, RET] — SUB需要2个操作数但栈空")
print(f"  结果: {result_bad}  (VM 返回 None)")

公式1: abs(RET - VOL)
  Token: [0, 1, 7, 11]
  结果形状: [5, 100]

公式2: sign(RET * FOMO)
  Token: [0, 3, 8, 12]
  含义: 收益率和FOMO同向时=+1 (做多), 反向时=-1

公式3: if PRESSURE>0 then RET else DEV
  Token: [2, 0, 4, 13]
  含义: 阳线时看收益率动量, 阴线时看偏离度

无效公式: [SUB, RET] — SUB需要2个操作数但栈空
  结果: None  (VM 返回 None)


## 3.4 AlphaGPT Transformer 架构

```
输入: [START_TOKEN]
  ↓
Token Embedding (17维 → 64维) + Positional Encoding
  ↓
Looped Transformer (2层, 每层循环3次)
├─ QKNorm 注意力 (Query-Key 独立归一化)
├─ RMSNorm (比 LayerNorm 更快更稳)
└─ SwiGLU 激活 (比 ReLU 更强)
  ↓
MTPHead (3个任务头加权融合)
  ↓
输出: 17维 logits (每个 Token 的概率)
```

### Looped Transformer 的特殊之处

```
普通Transformer:  Layer1 → Layer2 → Layer3     (3层, 3组参数)
Looped版本:       Layer1 → Layer1 → Layer1 →    (2层, 每层循环3次)
                  Layer2 → Layer2 → Layer2      共2组参数, 效果类似6层

好处: 参数更少，不容易过拟合，适合小词汇表
```

In [25]:
# 实例化模型并查看结构
from model_core.alphagpt import AlphaGPT

model = AlphaGPT()

# 模型基本信息
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== AlphaGPT 模型信息 ===")
print(f"词汇表: {model.vocab}")
print(f"词汇大小: {model.vocab_size}")
print(f"嵌入维度: {model.d_model}")
print(f"总参数量: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")

print("\n模型结构:")
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<15} → {params:>6,} 参数")

=== AlphaGPT 模型信息 ===
词汇表: ['RET', 'VOL', 'V_CHG', 'PV', 'TREND', 'ADD', 'SUB', 'MUL', 'DIV', 'NEG', 'ABS', 'SIGN', 'GATE', 'JUMP', 'DECAY', 'DELAY1', 'MAX3']
词汇大小: 17
嵌入维度: 64
总参数量: 90,906
可训练参数: 90,906

模型结构:
  token_emb       →  1,088 参数
  blocks          → 83,360 参数
  ln_f            →     64 参数
  mtp_head        →  5,497 参数
  head_critic     →     65 参数


In [26]:
# 模型前向传播演示

# 输入: 一个公式的前缀 (batch_size=2, seq_len=3)
# 假设已生成 [START=0, RET=0, VOL=1]，预测下一个 Token
input_tokens = torch.tensor([[0, 0, 1],
                              [0, 3, 4]])  # 两个不同的公式前缀

print("=== 模型前向传播 ===")
print(f"输入形状: {list(input_tokens.shape)} — [batch=2, seq_len=3]")

with torch.no_grad():
    logits, value, task_probs = model(input_tokens)

print(f"\n输出:")
print(f"  logits:     {list(logits.shape)} — 每个 Token 的得分")
print(f"  value:      {list(value.shape)} — 状态评估值")
print(f"  task_probs: {list(task_probs.shape)} — 3个任务头的权重")

# 查看第一个公式的预测
probs = torch.softmax(logits[0], dim=0)
top5 = torch.topk(probs, 5)

vocab = model.vocab
print(f"\n公式前缀 [START, RET, VOL] → 下一个 Token 预测:")
for prob, idx in zip(top5.values, top5.indices):
    print(f"  {vocab[idx]:>6} (Token {idx:>2}): {prob:.4f} ({prob*100:.1f}%)")

print(f"\n任务头权重: {task_probs[0].tolist()}")
print("  (3个子模型各自的投票权重)")

=== 模型前向传播 ===
输入形状: [2, 3] — [batch=2, seq_len=3]

输出:
  logits:     [2, 17] — 每个 Token 的得分
  value:      [2, 1] — 状态评估值
  task_probs: [2, 3] — 3个任务头的权重

公式前缀 [START, RET, VOL] → 下一个 Token 预测:
     MUL (Token  7): 0.0998 (10.0%)
    SIGN (Token 11): 0.0886 (8.9%)
   DECAY (Token 14): 0.0881 (8.8%)
   TREND (Token  4): 0.0852 (8.5%)
    JUMP (Token 13): 0.0780 (7.8%)

任务头权重: [0.3777749538421631, 0.2570945918560028, 0.3651304543018341]
  (3个子模型各自的投票权重)


## 3.5 MTPHead 多任务头

3个任务头独立预测，然后用路由器动态加权：

```
Head 1 → logits_1   (偏好动量型公式)
Head 2 → logits_2   (偏好均值回归型公式)       Router
Head 3 → logits_3   (偏好波动率型公式)     →   加权融合

类比: 三个基金经理各提建议，董事会按权重投票
```

In [29]:
# 演示 MTPHead 的工作原理
from model_core.alphagpt import MTPHead
import torch.nn.functional as F

# 模拟 3 个任务头的输出
d_model = 64
vocab_size = 17
mtp = MTPHead(d_model, vocab_size, num_tasks=3)

# 模拟输入 (最后一个 token 的嵌入)
x = torch.randn(1, d_model)

with torch.no_grad():
    logits, task_probs = mtp(x)

print("=== MTPHead 多任务头 ===")
print(f"输入: 嵌入向量 shape={list(x.shape)}")
print(f"\n3个任务头的权重: {task_probs[0].tolist()}")
print(f"  (由 Router 网络动态计算, 加权和=1)")
print(f"\n最终输出: logits shape={list(logits.shape)} — 17个Token的得分")

probs = F.softmax(logits[0], dim=0)
print(f"\nTop 3 预测:")
top3 = torch.topk(probs, 3)
for p, idx in zip(top3.values, top3.indices):
    token_name = features_list[idx] if idx < 5 else OPS_CONFIG[idx-5][0]
    print(f"  {token_name:>6}: {p:.4f}")

=== MTPHead 多任务头 ===
输入: 嵌入向量 shape=[1, 64]

3个任务头的权重: [0.35651782155036926, 0.27947738766670227, 0.3640047311782837]
  (由 Router 网络动态计算, 加权和=1)

最终输出: logits shape=[1, 17] — 17个Token的得分

Top 3 预测:
     ADD: 0.1077
     VOL: 0.0915
   V_CHG: 0.0821


---
# 阶段 4: 训练与回测

**学习目标**: 理解强化学习训练循环和回测评分机制

**涉及文件**:
- `model_core/engine.py` — 训练引擎
- `model_core/backtest.py` — 回测评分

## 4.1 训练的核心思想

```
试错法 (Policy Gradient / REINFORCE):

1. 随机生成 8192 个公式
2. 每个公式用 StackVM 执行，得到信号
3. 每个信号用 MemeBacktest 回测打分
4. 好公式 → 增大其生成概率
   差公式 → 降低其生成概率
5. 重复 1000 轮
```

类比：8192只猴子随机写交易策略，表现好的猴子得到香蕉，下一轮模仿它们的写法。

In [31]:
# 演示训练的一个步骤
from torch.distributions import Categorical
from model_core.config import ModelConfig
from model_core.vm import StackVM  # 添加这一行


model = AlphaGPT()
vm = StackVM()

# 小批量演示 (正式训练用 8192)
DEMO_BATCH = 4
MAX_LEN = 6  # 公式最大长度

print("=== 训练单步演示 ===")
print(f"批大小: {DEMO_BATCH}, 最大公式长度: {MAX_LEN}\n")

# Step 1: 采样公式
print("--- Step 1: 采样公式 ---")
inp = torch.zeros((DEMO_BATCH, 1), dtype=torch.long)  # START token
log_probs_all = []
formulas = []

with torch.no_grad():
    for pos in range(MAX_LEN):
        logits, _, _ = model(inp)
        dist = Categorical(logits=logits)
        action = dist.sample()  # 按概率采样
        log_probs_all.append(dist.log_prob(action))
        inp = torch.cat([inp, action.unsqueeze(1)], dim=1)

seqs = inp[:, 1:]  # 去掉 START token

vocab = model.vocab
for i in range(DEMO_BATCH):
    tokens = seqs[i].tolist()
    readable = [vocab[t] for t in tokens]
    print(f"  公式 {i}: {tokens} → {readable}")

=== 训练单步演示 ===
批大小: 4, 最大公式长度: 6

--- Step 1: 采样公式 ---
  公式 0: [15, 7, 12, 13, 4, 16] → ['DELAY1', 'MUL', 'GATE', 'JUMP', 'TREND', 'MAX3']
  公式 1: [2, 0, 13, 0, 15, 9] → ['V_CHG', 'RET', 'JUMP', 'RET', 'DELAY1', 'NEG']
  公式 2: [6, 9, 3, 0, 3, 7] → ['SUB', 'NEG', 'PV', 'RET', 'PV', 'MUL']
  公式 3: [15, 15, 0, 15, 1, 11] → ['DELAY1', 'DELAY1', 'RET', 'DELAY1', 'VOL', 'SIGN']


In [32]:
# Step 2: 执行公式并评分
print("--- Step 2: 执行公式并评分 ---\n")

from model_core.backtest import MemeBacktest
bt = MemeBacktest()

rewards = []
for i in range(DEMO_BATCH):
    formula = seqs[i].tolist()
    readable = [vocab[t] for t in formula]
    
    # 执行公式
    result = vm.execute(formula, feat_tensor)
    
    if result is None:
        reward = -5.0
        print(f"  公式 {i}: {readable}")
        print(f"    → 执行失败 (操作数不足), reward = {reward}")
    elif result.std() < 1e-4:
        reward = -2.0
        print(f"  公式 {i}: {readable}")
        print(f"    → 信号太平 (没有区分度), reward = {reward}")
    else:
        score, ret_val = bt.evaluate(result, raw_data, target_ret)
        reward = score.item()
        print(f"  公式 {i}: {readable}")
        print(f"    → score={reward:.4f}, cum_ret={ret_val:.2%}")
    
    rewards.append(reward)

rewards = torch.tensor(rewards)
print(f"\n所有 reward: {rewards.tolist()}")

--- Step 2: 执行公式并评分 ---

  公式 0: ['DELAY1', 'MUL', 'GATE', 'JUMP', 'TREND', 'MAX3']
    → 执行失败 (操作数不足), reward = -5.0
  公式 1: ['V_CHG', 'RET', 'JUMP', 'RET', 'DELAY1', 'NEG']
    → 执行失败 (操作数不足), reward = -5.0
  公式 2: ['SUB', 'NEG', 'PV', 'RET', 'PV', 'MUL']
    → 执行失败 (操作数不足), reward = -5.0
  公式 3: ['DELAY1', 'DELAY1', 'RET', 'DELAY1', 'VOL', 'SIGN']
    → 执行失败 (操作数不足), reward = -5.0

所有 reward: [-5.0, -5.0, -5.0, -5.0]


In [33]:
# Step 3: 策略梯度更新
print("--- Step 3: 策略梯度 (Advantage 计算) ---\n")

# 优势函数 = (reward - 均值) / 标准差
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-5)

for i in range(DEMO_BATCH):
    r = rewards[i].item()
    a = advantages[i].item()
    effect = "增大概率" if a > 0 else "降低概率"
    print(f"  公式 {i}: reward={r:+.4f} → advantage={a:+.4f} → {effect}")

print("\n核心公式:")
print("  loss = -sum(log_prob * advantage)")
print("  advantage > 0 → -log_prob * (+) → loss 减小 → 概率增大")
print("  advantage < 0 → -log_prob * (-) → loss 增大 → 概率减小")

--- Step 3: 策略梯度 (Advantage 计算) ---

  公式 0: reward=-5.0000 → advantage=+0.0000 → 降低概率
  公式 1: reward=-5.0000 → advantage=+0.0000 → 降低概率
  公式 2: reward=-5.0000 → advantage=+0.0000 → 降低概率
  公式 3: reward=-5.0000 → advantage=+0.0000 → 降低概率

核心公式:
  loss = -sum(log_prob * advantage)
  advantage > 0 → -log_prob * (+) → loss 减小 → 概率增大
  advantage < 0 → -log_prob * (-) → loss 增大 → 概率减小


## 4.2 回测评分详解

MemeBacktest 把公式输出的信号转为模拟交易，计算盈亏。

In [34]:
# 回测评分流程演示

# 用一个简单公式生成信号
signal_raw = vm.execute([0, 4, 7, 12], feat_tensor)  # sign(RET - TREND)

if signal_raw is not None:
    print("=== 回测评分流程 ===\n")
    
    # Step 1: sigmoid 把信号转为概率
    signal = torch.sigmoid(signal_raw)
    print(f"Step 1: sigmoid 转换")
    print(f"  原始信号范围: [{signal_raw.min():.3f}, {signal_raw.max():.3f}]")
    print(f"  转换后范围:   [{signal.min():.3f}, {signal.max():.3f}]  (0~1之间)")
    
    # Step 2: 生成仓位 (>0.85 且流动性安全才持仓)
    liq = raw_data['liquidity']
    is_safe = (liq > 500000.0).float()
    position = (signal > 0.85).float() * is_safe
    pct_holding = position.sum().item() / position.numel() * 100
    print(f"\nStep 2: 生成仓位")
    print(f"  阈值: signal > 0.85 且 liquidity > 50万")
    print(f"  持仓比例: {pct_holding:.1f}% 的时间段在持仓")
    
    # Step 3: 计算成本
    prev_pos = torch.roll(position, 1, dims=1)
    prev_pos[:, 0] = 0
    turnover = torch.abs(position - prev_pos)
    base_fee = 0.006
    impact = torch.clamp(1000.0 / (liq + 1e-9), 0, 0.05)
    tx_cost = turnover * (base_fee + impact)
    avg_cost = tx_cost[tx_cost > 0].mean().item() if (tx_cost > 0).any() else 0
    print(f"\nStep 3: 交易成本")
    print(f"  基础费率: {base_fee*100:.1f}%")
    print(f"  平均单次成本: {avg_cost*100:.3f}%")
    
    # Step 4: 计算PnL
    gross_pnl = position * target_ret
    net_pnl = gross_pnl - tx_cost
    cum_ret = net_pnl.sum(dim=1)
    print(f"\nStep 4: 盈亏统计")
    print(f"  各股票累计收益: {cum_ret.tolist()}")
    
    # Step 5: 最终评分
    big_dd = (net_pnl < -0.05).float().sum(dim=1)
    score = cum_ret - big_dd * 2.0
    activity = position.sum(dim=1)
    score = torch.where(activity < 5, torch.tensor(-10.0), score)
    final = torch.median(score)
    print(f"\nStep 5: 最终评分")
    print(f"  大回撤惩罚: 每次 net_pnl < -5% 扣 2 分")
    print(f"  低活跃惩罚: 持仓天数 < 5 则 score = -10")
    print(f"  最终适应度: median(score) = {final:.4f}")
else:
    print("公式执行失败，请尝试其他公式")

=== 回测评分流程 ===

Step 1: sigmoid 转换
  原始信号范围: [-1.000, 1.000]
  转换后范围:   [0.269, 0.731]  (0~1之间)

Step 2: 生成仓位
  阈值: signal > 0.85 且 liquidity > 50万
  持仓比例: 0.0% 的时间段在持仓

Step 3: 交易成本
  基础费率: 0.6%
  平均单次成本: 0.000%

Step 4: 盈亏统计
  各股票累计收益: [0.0, 0.0, 0.0, 0.0, 0.0]

Step 5: 最终评分
  大回撤惩罚: 每次 net_pnl < -5% 扣 2 分
  低活跃惩罚: 持仓天数 < 5 则 score = -10
  最终适应度: median(score) = -10.0000


## 4.3 LoRD 正则化

**问题**: Transformer 容易过拟合——记住历史数据，但预测未来不行  
**解决**: Low-Rank Decay (低秩衰减)

```
没有 LoRD → 模型学到 "2024年3月15日平安银行会涨"    (记忆)
有了 LoRD → 模型学到 "量价齐升时买入"              (规律)

原理：强迫注意力矩阵保持"低秩" = 只学简单规律，不记复杂细节
```

In [35]:
# 演示 Stable Rank (稳定秩) 的概念
from model_core.alphagpt import StableRankMonitor

monitor = StableRankMonitor(model)

# 初始稳定秩
rank_before = monitor.compute()
print("=== Stable Rank (稳定秩) ===")
print(f"\n公式: Stable Rank = ||W||_F^2 / ||W||_2^2")
print(f"  ||W||_F = Frobenius范数 (所有元素平方和的根)")
print(f"  ||W||_2 = 谱范数 (最大奇异值)")
print(f"\n初始平均稳定秩: {rank_before:.2f}")
print(f"\n解读:")
print(f"  秩 = 1 → 矩阵退化成一维，信息太少")
print(f"  秩很高 → 矩阵太复杂，容易过拟合")
print(f"  LoRD 的作用: 每步轻微推动秩向低处走，保持简洁")

=== Stable Rank (稳定秩) ===

公式: Stable Rank = ||W||_F^2 / ||W||_2^2
  ||W||_F = Frobenius范数 (所有元素平方和的根)
  ||W||_2 = 谱范数 (最大奇异值)

初始平均稳定秩: 22.25

解读:
  秩 = 1 → 矩阵退化成一维，信息太少
  秩很高 → 矩阵太复杂，容易过拟合
  LoRD 的作用: 每步轻微推动秩向低处走，保持简洁


## 4.4 完整训练（小规模演示）

下面用**缩小版参数**运行几步训练，感受整个流程：

In [36]:
# 小规模训练演示
from model_core.alphagpt import AlphaGPT, NewtonSchulzLowRankDecay
from model_core.vm import StackVM
from model_core.backtest import MemeBacktest
from torch.distributions import Categorical

# 初始化
model = AlphaGPT()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
lord = NewtonSchulzLowRankDecay(model.named_parameters(), decay_rate=1e-3)
vm = StackVM()
bt = MemeBacktest()

# 缩小参数
BATCH_SIZE = 32     # 正式训练用 8192
TRAIN_STEPS = 10    # 正式训练用 1000
MAX_LEN = 8         # 正式训练用 12

best_score = -float('inf')
best_formula = None

print(f"=== 小规模训练 (batch={BATCH_SIZE}, steps={TRAIN_STEPS}) ===")
print(f"词汇表: {model.vocab}\n")

for step in range(TRAIN_STEPS):
    inp = torch.zeros((BATCH_SIZE, 1), dtype=torch.long)
    log_probs = []
    
    # 采样
    for _ in range(MAX_LEN):
        logits, _, _ = model(inp)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        inp = torch.cat([inp, action.unsqueeze(1)], dim=1)
    
    seqs = inp[:, 1:]
    
    # 评分
    rewards = torch.zeros(BATCH_SIZE)
    for i in range(BATCH_SIZE):
        formula = seqs[i].tolist()
        res = vm.execute(formula, feat_tensor)
        if res is None:
            rewards[i] = -5.0
        elif res.std() < 1e-4:
            rewards[i] = -2.0
        else:
            score, ret_val = bt.evaluate(res, raw_data, target_ret)
            rewards[i] = score
            if score.item() > best_score:
                best_score = score.item()
                best_formula = formula
    
    # 更新
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-5)
    loss = sum(-lp * adv for lp in log_probs).mean()
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    lord.step()
    
    print(f"  Step {step}: avg_reward={rewards.mean():.3f}, best={best_score:.3f}")

if best_formula:
    readable = [model.vocab[t] for t in best_formula]
    print(f"\n最佳公式: {best_formula}")
    print(f"可读形式: {readable}")
    print(f"最佳得分: {best_score:.4f}")
else:
    print("\n未找到有效公式 (训练步数太少，正常)")

=== 小规模训练 (batch=32, steps=10) ===
词汇表: ['RET', 'VOL', 'V_CHG', 'PV', 'TREND', 'ADD', 'SUB', 'MUL', 'DIV', 'NEG', 'ABS', 'SIGN', 'GATE', 'JUMP', 'DECAY', 'DELAY1', 'MAX3']

  Step 0: avg_reward=-4.998, best=0.050
  Step 1: avg_reward=-5.000, best=0.050
  Step 2: avg_reward=-5.000, best=0.050
  Step 3: avg_reward=-4.696, best=0.055
  Step 4: avg_reward=-4.844, best=0.055
  Step 5: avg_reward=-4.845, best=0.055
  Step 6: avg_reward=-4.695, best=0.055
  Step 7: avg_reward=-5.156, best=0.055
  Step 8: avg_reward=-5.000, best=0.055
  Step 9: avg_reward=-5.000, best=0.055

最佳公式: [5, 4, 15, 10, 15, 6, 11, 15]
可读形式: ['ADD', 'TREND', 'DELAY1', 'ABS', 'DELAY1', 'SUB', 'SIGN', 'DELAY1']
最佳得分: 0.0550


---
# 阶段 5: 策略执行

**学习目标**: 理解从信号到交易的完整执行链路

**涉及文件**:
- `strategy_manager/runner.py` — 主循环
- `strategy_manager/portfolio.py` — 持仓管理
- `strategy_manager/risk.py` — 风险控制
- `strategy_manager/config.py` — 策略参数

## 5.1 策略参数

```python
MAX_OPEN_POSITIONS = 3     # 最多同时持有 3 只
ENTRY_AMOUNT_SOL = 2.0     # 每只投入 2 SOL（A股中对应固定金额）
STOP_LOSS_PCT = -0.05      # 止损: -5%
TAKE_PROFIT_Target1 = 0.10 # 第一止盈: +10% 时卖掉 50%
TRAILING_ACTIVATION = 0.05 # 追踪止损激活: +5%
TRAILING_DROP = 0.03       # 追踪回撤: -3%
BUY_THRESHOLD = 0.85       # 买入信号阈值
SELL_THRESHOLD = 0.45      # 卖出信号阈值
```

## 5.2 入场逻辑

In [37]:
# 模拟入场逻辑
print("=== 入场逻辑模拟 ===\n")

# 假设我们有一个训练好的公式
demo_formula = [0, 3, 8, 12]  # sign(RET * FOMO)
readable = [model.vocab[t] for t in demo_formula]
print(f"使用公式: {readable}\n")

# 执行公式，获取信号
raw_signal = vm.execute(demo_formula, feat_tensor)
if raw_signal is not None:
    # 只看最新一天的信号
    latest_signal = torch.sigmoid(raw_signal[:, -1])  # [N_stocks]
    
    stock_names = [f"Stock_{i}" for i in range(N_STOCKS)]
    
    # 排序
    sorted_idx = torch.argsort(latest_signal, descending=True)
    
    BUY_THRESHOLD = 0.85
    MAX_POSITIONS = 3
    positions = []
    
    print(f"{'排名':>4} {'股票':>10} {'信号':>8} {'决策':>10}")
    print("-" * 40)
    for rank, idx in enumerate(sorted_idx):
        score = latest_signal[idx].item()
        if score >= BUY_THRESHOLD and len(positions) < MAX_POSITIONS:
            decision = "买入"
            positions.append(idx.item())
        elif score >= BUY_THRESHOLD:
            decision = "满仓跳过"
        else:
            decision = "信号不足"
        print(f"  {rank+1:>2}  {stock_names[idx]:>10}  {score:>7.4f}  {decision:>8}")
    
    print(f"\n最终持仓: {[stock_names[i] for i in positions]}")
else:
    print("公式执行失败")

=== 入场逻辑模拟 ===

使用公式: ['RET', 'PV', 'DIV', 'GATE']

  排名         股票       信号         决策
----------------------------------------
   1     Stock_0   0.7311      信号不足
   2     Stock_1   0.7311      信号不足
   3     Stock_2   0.7311      信号不足
   4     Stock_3   0.7311      信号不足
   5     Stock_4   0.7311      信号不足

最终持仓: []


In [38]:
# 模拟出场逻辑
print("=== 出场逻辑模拟 ===\n")

# 模拟一个持仓的价格变化
entry_price = 11.0
price_series = [11.0, 11.3, 11.8, 12.1, 12.5, 13.0, 12.8, 12.5, 12.2]

STOP_LOSS = -0.05
TAKE_PROFIT_1 = 0.10
TRAILING_ACTIVATION = 0.05
TRAILING_DROP = 0.03

is_moonbag = False
highest_price = entry_price
holding_pct = 1.0  # 100% 持仓

print(f"入场价: {entry_price}")
print(f"止损线: {entry_price * (1 + STOP_LOSS):.2f} (-5%)")
print(f"止盈线: {entry_price * (1 + TAKE_PROFIT_1):.2f} (+10%)\n")

print(f"{'Day':>4} {'Price':>7} {'PnL':>8} {'Highest':>8} {'Action':>20} {'Holding':>10}")
print("-" * 65)

for day, price in enumerate(price_series):
    if holding_pct <= 0:
        break
    
    pnl = (price - entry_price) / entry_price
    highest_price = max(highest_price, price)
    action = "持有"
    
    # 检查止损
    if pnl <= STOP_LOSS:
        action = "止损 (全部卖出)"
        holding_pct = 0
    # 检查第一止盈
    elif pnl >= TAKE_PROFIT_1 and not is_moonbag:
        action = "止盈1 (卖出50%)"
        holding_pct = 0.5
        is_moonbag = True
    # 检查追踪止损
    elif (highest_price - entry_price) / entry_price > TRAILING_ACTIVATION:
        drawdown = (highest_price - price) / highest_price
        if drawdown > TRAILING_DROP:
            action = f"追踪止损 (回撤{drawdown:.1%})"
            holding_pct = 0
    
    print(f"  {day:>2}  {price:>7.2f}  {pnl:>+7.2%}  {highest_price:>7.2f}  {action:>20}  {holding_pct:>8.0%}")

=== 出场逻辑模拟 ===

入场价: 11.0
止损线: 10.45 (-5%)
止盈线: 12.10 (+10%)

 Day   Price      PnL  Highest               Action    Holding
-----------------------------------------------------------------
   0    11.00   +0.00%    11.00                    持有      100%
   1    11.30   +2.73%    11.30                    持有      100%
   2    11.80   +7.27%    11.80                    持有      100%
   3    12.10  +10.00%    12.10                    持有      100%
   4    12.50  +13.64%    12.50           止盈1 (卖出50%)       50%
   5    13.00  +18.18%    13.00                    持有       50%
   6    12.80  +16.36%    13.00                    持有       50%
   7    12.50  +13.64%    13.00         追踪止损 (回撤3.8%)        0%


## 5.3 风险管理

```
三层防护:

第1层: 流动性检查
  日成交额 < 50万 → 拒绝买入 (买得进去卖不出来)

第2层: 仓位控制
  最多同时 3 只, 每只固定金额 (不 all-in)

第3层: 止损止盈
  -5% 硬止损 → 绝不扛单
  +10% 减半  → 落袋为安
  追踪止损   → 保护利润
```

---
# 总结 & 练习

## 系统全流程回顾

```
1. Tushare API → 获取沪深300成分股 + 日线数据
2. PostgreSQL  → 存储 tokens 表 + ohlcv 时序表
3. 特征工程    → 6维: [RET, LIQ, PRESSURE, FOMO, DEV, LOG_VOL]
4. AlphaGPT    → Looped Transformer 生成公式 Token 序列
5. StackVM     → 执行公式，产生 [N_stocks, T_days] 信号矩阵
6. Backtest    → sigmoid(signal) > 0.85 → 持仓 → 计算PnL → 评分
7. REINFORCE   → 好公式增大概率，差公式降低概率，迭代1000轮
8. 实盘        → 加载最佳公式 → 每日扫描入场 → 风控出场
```

## 动手练习

### 练习1: 设计自己的公式
尝试手写不同的 Token 序列，看回测得分如何。

In [41]:
# 练习1: 修改下面的公式，尝试不同组合

# 词汇表提醒:
# 特征: 0=RET, 1=VOL, 2=V_CHG, 3=PV, 4=TREND
# 操作: 5=ADD, 6=SUB, 7=MUL, 8=DIV, 9=NEG, 10=ABS
#        11=SIGN, 12=GATE, 13=JUMP, 14=DECAY, 15=DELAY1, 16=MAX3

my_formula = [1, 16, 0, 6, 11]  # sign(RET - DELAY1(RET)) = 动量反转
# ↑↑↑ 修改这里试试不同公式 ↑↑↑

readable = [model.vocab[t] if t < len(model.vocab) else '?' for t in my_formula]
print(f"公式: {my_formula}")
print(f"可读: {readable}\n")

result = vm.execute(my_formula, feat_tensor)
if result is not None:
    score, ret_val = bt.evaluate(result, raw_data, target_ret)
    print(f"适应度: {score:.4f}")
    print(f"累计收益: {ret_val:.2%}")
    
    # 持仓统计
    sig = torch.sigmoid(result)
    holding_pct = (sig > 0.85).float().mean().item() * 100
    print(f"持仓时间占比: {holding_pct:.1f}%")
else:
    print("公式无效! 检查操作数是否匹配")

公式: [1, 16, 0, 6, 11]
可读: ['VOL', 'MAX3', 'RET', 'SUB', 'SIGN']

适应度: -0.1204
累计收益: -19.63%
持仓时间占比: 25.6%


### 练习2: 修改回测参数
调整费率和阈值，观察对结果的影响。

In [42]:
# 练习2: 修改回测参数

class MyBacktest:
    """自定义回测器 — 修改参数看看效果"""
    def __init__(self, base_fee=0.006, threshold=0.85, min_liq=500000):
        self.trade_size = 1000.0
        self.min_liq = min_liq
        self.base_fee = base_fee
        self.threshold = threshold
    
    def evaluate(self, factors, raw_data, target_ret):
        liq = raw_data['liquidity']
        signal = torch.sigmoid(factors)
        is_safe = (liq > self.min_liq).float()
        position = (signal > self.threshold).float() * is_safe
        
        impact = torch.clamp(self.trade_size / (liq + 1e-9), 0, 0.05)
        prev_pos = torch.roll(position, 1, dims=1)
        prev_pos[:, 0] = 0
        turnover = torch.abs(position - prev_pos)
        tx_cost = turnover * (self.base_fee + impact)
        
        net_pnl = position * target_ret - tx_cost
        cum_ret = net_pnl.sum(dim=1)
        big_dd = (net_pnl < -0.05).float().sum(dim=1)
        score = cum_ret - big_dd * 2.0
        activity = position.sum(dim=1)
        score = torch.where(activity < 5, torch.tensor(-10.0), score)
        return torch.median(score), cum_ret.mean().item()

# 用一个公式测试不同参数
test_formula = [0, 14, 11]  # sign(decay(RET))
result = vm.execute(test_formula, feat_tensor)

if result is not None:
    configs = [
        {"name": "默认 (0.6%, 阈值0.85)", "fee": 0.006, "threshold": 0.85},
        {"name": "低费率 (0.1%)",          "fee": 0.001, "threshold": 0.85},
        {"name": "低阈值 (0.60)",          "fee": 0.006, "threshold": 0.60},
        {"name": "高阈值 (0.95)",          "fee": 0.006, "threshold": 0.95},
    ]
    
    print(f"公式: sign(decay(RET))\n")
    print(f"{'参数配置':<25} {'适应度':>8} {'累计收益':>10}")
    print("-" * 48)
    for cfg in configs:
        my_bt = MyBacktest(base_fee=cfg['fee'], threshold=cfg['threshold'])
        score, ret = my_bt.evaluate(result, raw_data, target_ret)
        print(f"{cfg['name']:<25} {score.item():>+8.4f} {ret:>+9.2%}")
    
    print("\n观察: 费率和阈值对结果影响巨大!")
    print("  低费率 → 更多交易更赚钱 (但现实中费率是固定的)")
    print("  低阈值 → 更频繁交易 (但噪音也更多)")
    print("  高阈值 → 更少交易 (可能错过机会)")

公式: sign(decay(RET))

参数配置                           适应度       累计收益
------------------------------------------------
默认 (0.6%, 阈值0.85)         -10.0000    +0.00%
低费率 (0.1%)                -10.0000    +0.00%
低阈值 (0.60)                -10.0000    +0.00%
高阈值 (0.95)                -10.0000    +0.00%

观察: 费率和阈值对结果影响巨大!
  低费率 → 更多交易更赚钱 (但现实中费率是固定的)
  低阈值 → 更频繁交易 (但噪音也更多)
  高阈值 → 更少交易 (可能错过机会)


### 练习3: 运行真实数据管道（需要数据库和Tushare Token）

In [ ]:
# 练习3: 运行真实数据管道（取消注释后运行）
# 前提: 
#   1. PostgreSQL 已启动
#   2. .env 中配置了 TUSHARE_TOKEN 和数据库连接

# import asyncio
# from data_pipeline.data_manager import DataManager
# from data_pipeline.config import DataSourceMode
#
# async def run_pipeline():
#     manager = DataManager(mode=DataSourceMode.ASTOCK)
#     try:
#         await manager.initialize()
#         await manager.pipeline_sync_daily()
#     finally:
#         await manager.close()
#
# await run_pipeline()

print("取消上面代码的注释即可运行真实数据管道")
print("\n命令行方式:")
print("  python -m data_pipeline.run_pipeline --mode astock --pool hs300")

### 练习4: 正式模型训练（需要数据库中有数据）

In [ ]:
# 练习4: 正式训练（取消注释后运行）
# 前提: 已运行过数据管道，数据库中有 ohlcv 数据

# from model_core.engine import AlphaEngine
# 
# engine = AlphaEngine(use_lord_regularization=True)
# engine.train()
# 
# # 训练完成后查看结果
# print(f"最佳公式: {engine.best_formula}")
# readable = [engine.model.vocab[t] for t in engine.best_formula]
# print(f"可读形式: {readable}")
# print(f"最佳得分: {engine.best_score:.4f}")

print("取消上面代码的注释即可运行正式训练")
print("\n命令行方式:")
print("  python -m model_core.engine")

---
## 附录: 关键文件速查表

| 阶段 | 文件 | 核心类/函数 |
|------|------|-------------|
| 配置 | `data_pipeline/config.py` | `Config`, `DataSourceMode` |
| 数据获取 | `data_pipeline/providers/tushare.py` | `TushareProvider` |
| 数据库 | `data_pipeline/db_manager.py` | `DBManager` |
| 管道编排 | `data_pipeline/data_manager.py` | `DataManager` |
| 数据加载 | `model_core/data_loader.py` | `CryptoDataLoader` |
| 特征工程 | `model_core/factors.py` | `FeatureEngineer`, `MemeIndicators` |
| 操作符 | `model_core/ops.py` | `OPS_CONFIG` |
| 虚拟机 | `model_core/vm.py` | `StackVM` |
| 模型 | `model_core/alphagpt.py` | `AlphaGPT`, `LoopedTransformer`, `MTPHead` |
| 训练 | `model_core/engine.py` | `AlphaEngine` |
| 回测 | `model_core/backtest.py` | `MemeBacktest` |
| 策略 | `strategy_manager/runner.py` | `StrategyRunner` |
| 持仓 | `strategy_manager/portfolio.py` | `PortfolioManager`, `Position` |
| 风控 | `strategy_manager/risk.py` | `RiskEngine` |